### Query Translation - Decomposition
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

### What is Query Decomposition?
**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It’s usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    “Decompose this question into a list of simpler search queries.”
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

#### Example

**User asks:**

    “Summarize Lilian Weng’s post on LLM agents. Focus on task decomposition methods and explain how Tree of Thoughts extends Chain of Thought.”

A **single vector search** might fail because:
* The query contains **multiple sub-topics** (“task decomposition methods” + “Tree of Thoughts vs CoT”).
* The embedding may dilute meaning across the whole sentence.
**Decomposition:**
* “What are task decomposition methods in Lilian Weng’s LLM agents post?”
* “How does Tree of Thoughts extend Chain of Thought reasoning?”

Retrieve separately, then combine into a coherent answer.

### Why / When to Use Query Decomposition

✅ Use it when:
* **Complex / multi-aspect questions:** 
    e.g., “Compare AutoGPT and BabyAGI, and explain how planning differs from memory.”

* **Broad tasks spanning sub-topics:**
    e.g., “Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion.”

* **Long, natural language queries:** with multiple clauses joined by “and”, “or”, “how … and also …”.

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

🚫 Less helpful when:
* The query is **short and atomic** (e.g., “What is RAG Fusion?”).
* The corpus is tiny or each document already covers the entire topic.

In [22]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [23]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [24]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

In [25]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [26]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_qd

As a first step, let us ask the LLM our question & check it's response without any query decomposition.

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [28]:
QUESTION = "What is task decomposition for LLM agents?"

In [29]:
template = ChatPromptTemplate.from_template("Answer the following question:\n\n{question}")
simple_chain = template | llm | StrOutputParser()
response = simple_chain.invoke({"question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down a complex, high-level goal or task into a series 
of smaller, more manageable, and actionable sub-tasks or steps.                                                    

This process is crucial because while Large Language Models (LLMs) are powerful, they have limitations, especially 
when faced with multi-step reasoning, long chains of thought, or tasks requiring extensive planning and execution. 

Here's a breakdown of what it entails and why it's important:                                                      

                                Why is Task Decomposition Necessary for LLM Agents?                                

 1 Overcoming Context Window Limitations: Complex tasks often require more information and intermediate thoughts   
   than can fit into a single LLM prompt's context window. Decomposing allows the agent to process information in  
   smaller, focused chunks.                                                                                        
 2 Improving Accuracy and Reliability: LLMs are more likely to perform well on simpler, well-defined sub-tasks.    
   Breaking down a task reduces the cognitive load on the LLM, minimizing the chances of hallucination, errors, or 
   getting "lost" in a complex prompt.                                                                             
 3 Enabling Multi-Step Reasoning: Many real-world tasks require sequential steps, where the output of one step     
   informs the next. Decomposition naturally facilitates this step-by-step reasoning.                              
 4 Facilitating Tool Use: Each sub-task can be designed to leverage specific external tools (e.g., web search, code
   interpreter, API calls, database queries) more effectively. The agent can decide which tool is appropriate for  
   each specific sub-task.                                                                                         
 5 Enhanced Debugging and Error Handling: If an agent fails to complete a complex task, it's hard to pinpoint where
   the error occurred. With decomposition, failures can be isolated to specific sub-tasks, making debugging and    
   recovery much easier.                                                                                           
 6 Increased Transparency and Interpretability: By breaking down a task, the agent's thought process becomes more  
   explicit and understandable, as you can see the individual steps it takes.                                      
 7 Better Resource Management: For very long tasks, decomposition allows for more efficient use of computational   
   resources, as the agent can focus on one sub-task at a time.                                                    

                                   How Task Decomposition Works for LLM Agents:                                    

The LLM agent typically uses the LLM itself to perform the decomposition, often guided by specific prompting       
strategies.                                                                                                        

 1 Initial Prompting: The agent receives the main goal. It then prompts the LLM (or a specialized "planner" LLM)   
   with instructions like: "Given the following task, break it down into a sequence of logical, actionable steps.  
   For each step, consider what information is needed and what tools might be useful."                             
 2 LLM Generates Sub-tasks: The LLM responds with a list of sub-tasks. This might be a simple linear list, or a    
   more complex tree-like structure (e.g., Tree-of-Thought).                                                       
 3 Execution Loop: The agent then iterates through these sub-tasks:                                                
    • Select Sub-task: Choose the next sub-task to execute.                                                        
    • Context Management: Pass relevant informati

In [30]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or additional quotes around the queries or markdown text \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [31]:
queries_generator = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
questions = queries_generator.invoke(
    {
        "num_queries": 5,
        "question": QUESTION,
    }
)
console.print(questions)

[
    'define task decomposition LLM agents',
    'why is task decomposition important for LLM agents',
    'techniques for task decomposition in LLM agents',
    'how LLM agents perform task decomposition',
    'task decomposition strategies for large language models'
]

In [32]:
for q in questions:
    print(q)

define task decomposition LLM agents
why is task decomposition important for LLM agents
techniques for task decomposition in LLM agents
how LLM agents perform task decomposition
task decomposition strategies for large language models


So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

Once we have the decomposed questions, we'll tweak the way the LLM responds to these questions.

#### Answering Recursively
In this technique, the questions list we got above is passed recursively to the LLM - first Q1 is passed and we get a response A1 from LLM. Q1 + A1 is added as a context to Q2 to get A2, then (Q1 + A1) and (Q2 + A2) is added as a context when passing Q3 to the LLM and so on. Finally, we land up with context -> {(Q1, A1), (Q2, A2)...(QN-1, AN-1)} when QN is passed to the LLM. The answer from the LLM to QN with the above combined context is the final response. The intutition is by passing this "combined context" derived from the recursive process helps the LLM give a more coherent response to original question.

The image below illustrates this process visually:

![Multi Query](images/ans_recursively.png)

In [33]:
# Prompt
template = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [34]:
from operator import itemgetter


def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


q_a_pairs = ""
for i, q in enumerate(questions):
    rag_chain = (
        {
            # get the context by asking the retriver to retrieve it 
            # based on the question
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            # for the first question, q_a_pairs will be ""
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # format my prompt with above parameters
        | decomposition_prompt
        # ask LLM for response to formatted decomposition prompt
        | llm
        # parse out text as answer
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    console.print(f"[yellow]Intermediate QA-Pair #{i+1} -> [/yellow]")
    console.print(Markdown(q_a_pairs))

Intermediate QA-Pair #1 -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: define task decomposition LLM agents Answer: Task decomposition in LLM agents is a crucial planning      
component where a large, complicated task is broken down into smaller, more manageable subgoals or steps. This     
process enables the agent to efficiently handle complex tasks.                                                     

The LLM achieves this by being instructed to "think step by step," which helps transform hard tasks into simpler,  
actionable units. This approach is exemplified by techniques such as Chain of Thought (CoT), where the model       
explicitly outlines its reasoning steps, and Tree of Thoughts (ToT), which extends CoT by exploring multiple       
reasoning possibilities at each step, creating a tree structure of thoughts.                                       

Task decomposition can be performed by the LLM itself using simple prompts (e.g., "What are the subgoals for       
achieving XYZ?"), through task-specific instructions, or with human input.

Intermediate QA-Pair #2 -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: define task decomposition LLM agents Answer: Task decomposition in LLM agents is a crucial planning      
component where a large, complicated task is broken down into smaller, more manageable subgoals or steps. This     
process enables the agent to efficiently handle complex tasks.                                                     

The LLM achieves this by being instructed to "think step by step," which helps transform hard tasks into simpler,  
actionable units. This approach is exemplified by techniques such as Chain of Thought (CoT), where the model       
explicitly outlines its reasoning steps, and Tree of Thoughts (ToT), which extends CoT by exploring multiple       
reasoning possibilities at each step, creating a tree structure of thoughts.                                       


   Task decomposition can be performed by the LLM itself using simple prompts (e.g., "What are the subgoals for    
                    achieving XYZ?"), through task-specific instructions, or with human input.                     

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it allows them to effectively manage and execute complex, multi-step tasks.                                

Here's why it's important:                                                                                         

 1 Handles Complexity: Large, complicated tasks are difficult for LLMs to tackle directly. By breaking them down   
   into smaller, more manageable subgoals, agents can efficiently process and complete tasks that would otherwise  
   be overwhelming.                                                                                                
 2 Simplifies Tasks: It transforms hard problems into simpler, actionable units. This "think step by step"         
   approach, exemplified by Chain of Thought (CoT), enables the LLM to utilize more computation to solve problems  
   incrementally.                                                                                                  
 3 Enhances Performance: Decomposing tasks into smaller steps significantly enhances the model's performance on    
   complex tasks, as it allows for a structured approach to problem-solving.                                       
 4 Enables Planning: For tasks involving many steps, decomposition allows the agent to plan ahead, understanding   
   the sequence of actions required to achieve the overall goal.                                                   
 5 Improves Reasoning and Transparency: Techniques like CoT and Tree of Thoughts (ToT) not only help in breaking   
   down tasks but also shed light on the model's thinking process. ToT further enhances reasoning by exploring     
   multiple possibilities at each step, leading to more robust solutions.

Intermediate QA-Pair #3 -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: define task decomposition LLM agents Answer: Task decomposition in LLM agents is a crucial planning      
component where a large, complicated task is broken down into smaller, more manageable subgoals or steps. This     
process enables the agent to efficiently handle complex tasks.                                                     

The LLM achieves this by being instructed to "think step by step," which helps transform hard tasks into simpler,  
actionable units. This approach is exemplified by techniques such as Chain of Thought (CoT), where the model       
explicitly outlines its reasoning steps, and Tree of Thoughts (ToT), which extends CoT by exploring multiple       
reasoning possibilities at each step, creating a tree structure of thoughts.                                       


   Task decomposition can be performed by the LLM itself using simple prompts (e.g., "What are the subgoals for    
                    achieving XYZ?"), through task-specific instructions, or with human input.                     

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it allows them to effectively manage and execute complex, multi-step tasks.                                

Here's why it's important:                                                                                         

 1 Handles Complexity: Large, complicated tasks are difficult for LLMs to tackle directly. By breaking them down   
   into smaller, more manageable subgoals, agents can efficiently process and complete tasks that would otherwise  
   be overwhelming.                                                                                                
 2 Simplifies Tasks: It transforms hard problems into simpler, actionable units. This "think step by step"         
   approach, exemplified by Chain of Thought (CoT), enables the LLM to utilize more computation to solve problems  
   incrementally.                                                                                                  
 3 Enhances Performance: Decomposing tasks into smaller steps significantly enhances the model's performance on    
   complex tasks, as it allows for a structured approach to problem-solving.                                       
 4 Enables Planning: For tasks involving many steps, decomposition allows the agent to plan ahead, understanding   
   the sequence of actions required to achieve the overall goal.                                                   
 5 Improves Reasoning and Transparency: Techniques like CoT and Tree of Thoughts (ToT) not only help in breaking   
   down tasks but also shed light on the model's thinking process. ToT further enhances reasoning by exploring     
   multiple possibilities at each step, leading to more robust solutions.                                          

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: techniques for task decomposition in LLM agents Answer: Techniques for task decomposition in LLM agents  
primarily involve leveraging the LLM's reasoning capabilities to break down complex tasks into smaller, more       
manageable steps.                                                                                                  

The main techniques and methods include:                                                                           

 1 Chain of Thought (CoT): This is a standard prompting technique where the LLM is instructed to "think step by    
   step." This encourages the model to utilize more computation at test time to decompose hard tasks into simpler, 
   actionable steps. CoT transforms large tasks into multiple manageable sub-tasks and provides insight into the   
   model's reasoning process.                  

Intermediate QA-Pair #4 -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: define task decomposition LLM agents Answer: Task decomposition in LLM agents is a crucial planning      
component where a large, complicated task is broken down into smaller, more manageable subgoals or steps. This     
process enables the agent to efficiently handle complex tasks.                                                     

The LLM achieves this by being instructed to "think step by step," which helps transform hard tasks into simpler,  
actionable units. This approach is exemplified by techniques such as Chain of Thought (CoT), where the model       
explicitly outlines its reasoning steps, and Tree of Thoughts (ToT), which extends CoT by exploring multiple       
reasoning possibilities at each step, creating a tree structure of thoughts.                                       


   Task decomposition can be performed by the LLM itself using simple prompts (e.g., "What are the subgoals for    
                    achieving XYZ?"), through task-specific instructions, or with human input.                     

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it allows them to effectively manage and execute complex, multi-step tasks.                                

Here's why it's important:                                                                                         

 1 Handles Complexity: Large, complicated tasks are difficult for LLMs to tackle directly. By breaking them down   
   into smaller, more manageable subgoals, agents can efficiently process and complete tasks that would otherwise  
   be overwhelming.                                                                                                
 2 Simplifies Tasks: It transforms hard problems into simpler, actionable units. This "think step by step"         
   approach, exemplified by Chain of Thought (CoT), enables the LLM to utilize more computation to solve problems  
   incrementally.                                                                                                  
 3 Enhances Performance: Decomposing tasks into smaller steps significantly enhances the model's performance on    
   complex tasks, as it allows for a structured approach to problem-solving.                                       
 4 Enables Planning: For tasks involving many steps, decomposition allows the agent to plan ahead, understanding   
   the sequence of actions required to achieve the overall goal.                                                   
 5 Improves Reasoning and Transparency: Techniques like CoT and Tree of Thoughts (ToT) not only help in breaking   
   down tasks but also shed light on the model's thinking process. ToT further enhances reasoning by exploring     
   multiple possibilities at each step, leading to more robust solutions.                                          

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: techniques for task decomposition in LLM agents Answer: Techniques for task decomposition in LLM agents  
primarily involve leveraging the LLM's reasoning capabilities to break down complex tasks into smaller, more       
manageable steps.                                                                                                  

The main techniques and methods include:                                                                           

 1 Chain of Thought (CoT): This is a standard prompting technique where the LLM is instructed to "think step by    
   step." This encourages the model to utilize more computation at test time to decompose hard tasks into simpler, 
   actionable steps. CoT transforms large tasks into multiple manageable sub-tasks and provides insight into the   
   model's reasoning process.                  

Intermediate QA-Pair #5 -> 

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: define task decomposition LLM agents Answer: Task decomposition in LLM agents is a crucial planning      
component where a large, complicated task is broken down into smaller, more manageable subgoals or steps. This     
process enables the agent to efficiently handle complex tasks.                                                     

The LLM achieves this by being instructed to "think step by step," which helps transform hard tasks into simpler,  
actionable units. This approach is exemplified by techniques such as Chain of Thought (CoT), where the model       
explicitly outlines its reasoning steps, and Tree of Thoughts (ToT), which extends CoT by exploring multiple       
reasoning possibilities at each step, creating a tree structure of thoughts.                                       


   Task decomposition can be performed by the LLM itself using simple prompts (e.g., "What are the subgoals for    
                    achieving XYZ?"), through task-specific instructions, or with human input.                     

Question: why is task decomposition important for LLM agents Answer: Task decomposition is crucial for LLM agents  
because it allows them to effectively manage and execute complex, multi-step tasks.                                

Here's why it's important:                                                                                         

 1 Handles Complexity: Large, complicated tasks are difficult for LLMs to tackle directly. By breaking them down   
   into smaller, more manageable subgoals, agents can efficiently process and complete tasks that would otherwise  
   be overwhelming.                                                                                                
 2 Simplifies Tasks: It transforms hard problems into simpler, actionable units. This "think step by step"         
   approach, exemplified by Chain of Thought (CoT), enables the LLM to utilize more computation to solve problems  
   incrementally.                                                                                                  
 3 Enhances Performance: Decomposing tasks into smaller steps significantly enhances the model's performance on    
   complex tasks, as it allows for a structured approach to problem-solving.                                       
 4 Enables Planning: For tasks involving many steps, decomposition allows the agent to plan ahead, understanding   
   the sequence of actions required to achieve the overall goal.                                                   
 5 Improves Reasoning and Transparency: Techniques like CoT and Tree of Thoughts (ToT) not only help in breaking   
   down tasks but also shed light on the model's thinking process. ToT further enhances reasoning by exploring     
   multiple possibilities at each step, leading to more robust solutions.                                          

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: techniques for task decomposition in LLM agents Answer: Techniques for task decomposition in LLM agents  
primarily involve leveraging the LLM's reasoning capabilities to break down complex tasks into smaller, more       
manageable steps.                                                                                                  

The main techniques and methods include:                                                                           

 1 Chain of Thought (CoT): This is a standard prompting technique where the LLM is instructed to "think step by    
   step." This encourages the model to utilize more computation at test time to decompose hard tasks into simpler, 
   actionable steps. CoT transforms large tasks into multiple manageable sub-tasks and provides insight into the   
   model's reasoning process.                  

Notice how we keep adding a Q & A pair to the overall context. At the end of all the questions (Q&A pairs), we get the final answer from the LLM, which we will display below.

In [35]:
console.print(f"[yellow]Final answer:[/yellow]")
console.print(Markdown(answer))

Final answer:

Task decomposition strategies for large language models (LLMs) involve breaking down complex tasks into smaller,   
more manageable subgoals or steps. This process is crucial for LLMs to effectively handle multi-step problems and  
enhance their performance.                                                                                         

The primary strategies and methods include:                                                                        

 1 Chain of Thought (CoT): This is a widely used prompting technique where the LLM is explicitly instructed to     
   "think step by step." This encourages the model to utilize more computational effort to decompose a difficult   
   task into a sequence of simpler, actionable steps. CoT not only simplifies the task but also provides insight   
   into the model's reasoning process.                                                                             
 2 Tree of Thoughts (ToT): Extending the CoT approach, ToT explores multiple reasoning possibilities at each step  
   of the decomposition. It first breaks the problem into several "thought steps" and then generates multiple      
   potential "thoughts" for each step, forming a tree-like structure of possibilities. A search algorithm, such as 
   Breadth-First Search (BFS) or Depth-First Search (DFS), can then be employed to navigate this tree. Each state  
   (or thought) is evaluated, often by a classifier (via a prompt) or majority vote, to identify the most promising
   path towards the overall goal.                                                                                  

Beyond these specific reasoning techniques, task decomposition can be initiated and guided through several methods:

 • LLM with Simple Prompting: The LLM itself can perform the decomposition using straightforward prompts. Examples 
   include asking "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?".                              
 • Task-Specific Instructions: Providing the LLM with instructions tailored to the specific task can guide its     
   decomposition. For instance, instructing an agent to "Write a story outline" for a novel-writing task implicitly
   directs it to break down the larger goal into outline components.                                               
 • Human Inputs: Human users can directly provide input to guide or perform parts of the task decomposition,       
   offering explicit subgoals or steps to the LLM agent.

### Answer 
In this approach, we generate N different questions from the original question and fire them separately against the LLM. At the end, we combine all answers into a common context, based off which the final answer is generated by the LLM.

This is illustrated in image below:

![Ans Individually](images/ans_individually.png)

In [38]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.\n
    Question: {question} \n
    Context: {context} \n
    Answer:"""
)

In [42]:
def retrieve_individual_q_and_a(question, prompt_rag, sub_question_generator_chain, num_queries=5):
    # Use our decomposition / 
    sub_questions = sub_question_generator_chain.invoke({"question":question, 
                                                         "num_queries": num_queries})
    
    # Initialize a list to hold RAG chain results
    rag_results = []
    
    for sub_question in sub_questions:
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.get_relevant_documents(sub_question)
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs, 
                                                                "question": sub_question})
        rag_results.append(answer)
    
    return sub_questions, rag_results

questions, answers = retrieve_individual_q_and_a(QUESTION, rag_prompt, queries_generator)


In [43]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

q_n_a_pairs = format_qa_pairs(questions, answers)
print(q_n_a_pairs)

Question 1: define task decomposition LLM agents
Answer 1: Task decomposition in LLM agents is the process where a complex task is broken down into smaller, simpler, and more manageable steps or subgoals. This enables the agent to plan ahead and efficiently handle complicated tasks. It can be achieved by instructing the LLM to "think step by step" using prompting, task-specific instructions, or human input.

Question 2: why is task decomposition important for LLM agents
Answer 2: Task decomposition is crucial for LLM agents because complicated tasks involve many steps, requiring the agent to plan ahead. It transforms large, complex tasks into smaller, more manageable subgoals. This process enables the efficient handling of complex tasks and enhances the model's performance.

Question 3: techniques for task decomposition in LLM agents
Answer 3: Techniques for task decomposition in LLM agents include Chain of Thought (CoT), which instructs the model to "think step by step" to break down 

In [45]:
# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain.invoke({"context":q_n_a_pairs,"question":QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down a complex task into smaller, simpler, and more   
manageable steps or subgoals. This is a crucial strategy because complicated tasks often involve many steps,       
requiring the agent to plan ahead. By transforming large, complex tasks into smaller, more manageable subgoals, it 
enables the efficient handling of complex tasks and significantly enhances the model's performance.                

LLM agents achieve task decomposition through various techniques:                                                  

 1 Chain of Thought (CoT): This involves instructing the model to "think step by step," which prompts it to break  
   down problems into a sequence of smaller, logical steps.                                                        
 2 Tree of Thoughts (ToT): Extending CoT, ToT explores multiple reasoning possibilities or paths at each step,     
   generating a tree structure of thoughts. This allows for a more comprehensive search of potential solutions,    
   often using search algorithms like BFS or DFS.                                                                  
 3 Simple Prompting: Task decomposition can also be achieved through direct and simple instructions given to the   
   LLM.                                                                                                            
 4 Task-Specific Instructions: Providing the LLM with tailored instructions designed for a particular task can     
   guide it in breaking down the problem.                                                                          
 5 Human Input: Human intervention can directly guide the decomposition process, providing explicit subgoals or    
   steps for the LLM to follow.                                                                                    

In essence, task decomposition allows LLM agents to approach and solve intricate problems by systematically        
addressing their constituent parts, leading to more robust and effective task completion.